# Cell 1. 라이브러리 import 및 기본 경로 설정

In [1]:
import os
import re
import ast
import json
import gc
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image

from transformers import (
    AutoModelForCausalLM,
    AutoProcessor,
    AutoTokenizer,
    BitsAndBytesConfig
)

print("라이브러리 import 완료")
print("현재 작업 위치:", os.getcwd())

라이브러리 import 완료
현재 작업 위치: c:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\VLM_test


In [2]:
# 프레임 이미지 폴더
FRAMES_ROOT = Path("data/frames_for_vlm")

# 결과 저장 폴더
RESULT_DIR = Path("results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 8

# 결과 저장 파일
CSV_PATH = RESULT_DIR / "hyperclovax_vlm_1fps_chunked_results.csv"
JSON_PATH = RESULT_DIR / "hyperclovax_vlm_1fps_chunked_results.json"
FAILED_PATH = RESULT_DIR / "hyperclovax_vlm_1fps_chunked_failed.json"

# 모델 ID
MODEL_ID = "naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B"

# 출력 토큰 수
MAX_NEW_TOKENS = 384

print("CHUNK_SIZE:", CHUNK_SIZE)
print("FRAMES_ROOT 존재 여부:", FRAMES_ROOT.exists())
print("FRAMES_ROOT 절대 경로:", FRAMES_ROOT.resolve())
print("CSV_PATH:", CSV_PATH)
print("JSON_PATH:", JSON_PATH)

CHUNK_SIZE: 8
FRAMES_ROOT 존재 여부: True
FRAMES_ROOT 절대 경로: C:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\VLM_test\data\frames_for_vlm
CSV_PATH: results\hyperclovax_vlm_1fps_chunked_results.csv
JSON_PATH: results\hyperclovax_vlm_1fps_chunked_results.json


# Cell 2. GPU 상태 확인 및 이전 모델 정리

In [3]:
try:
    del model
except NameError:
    pass

try:
    del processor
except NameError:
    pass

try:
    del tokenizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("CUDA 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA cache 정리 완료")

CUDA 사용 가능 여부: True
GPU: NVIDIA GeForce RTX 3060 Ti
CUDA cache 정리 완료


# Cell 3. HyperCLOVAX 모델 로드

In [ ]:
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa"
).to("cuda")

model.eval()

# model.generation_config.do_sample = False
# model.generation_config.temperature = None
# model.generation_config.top_p = None
# model.generation_config.top_k = None

print("HyperCLOVAX fp16 모델 로드 완료")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

HyperCLOVAX fp16 모델 로드 완료


In [ ]:
# print("do_sample:", model.generation_config.do_sample)
# print("temperature:", model.generation_config.temperature)
# print("top_p:", model.generation_config.top_p)
# print("top_k:", model.generation_config.top_k)

# Cell 4. video_id 폴더 목록 확인

In [5]:
video_dirs = sorted([
    p for p in FRAMES_ROOT.iterdir()
    if p.is_dir()
])

print("영상 폴더 개수:", len(video_dirs))

for p in video_dirs:
    print("-", p.name)

영상 폴더 개수: 8
- knCQBlBKSRM
- LRmLOsmMHts
- mfi0n3oNABM
- MIJR3Dm0YOk
- NCVxpXxTMSU
- twi9zUxsXu0
- Wkiz8JVPvXA
- Xfu1kCID0Ls


# Cell 5. 프레임 이미지 수집 함수

In [6]:
def gather_images(video_folder: Path):
    """
    특정 video_id 폴더 안의 이미지 파일 경로를 정렬해서 가져온다.
    """
    image_paths = sorted([
        p for p in video_folder.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])
    return image_paths

In [7]:
for video_dir in video_dirs:
    image_paths = gather_images(video_dir)
    print(video_dir.name, "프레임 수:", len(image_paths))

knCQBlBKSRM 프레임 수: 21
LRmLOsmMHts 프레임 수: 30
mfi0n3oNABM 프레임 수: 56
MIJR3Dm0YOk 프레임 수: 50
NCVxpXxTMSU 프레임 수: 12
twi9zUxsXu0 프레임 수: 55
Wkiz8JVPvXA 프레임 수: 30
Xfu1kCID0Ls 프레임 수: 23


In [8]:
def split_into_chunks(items, chunk_size=8):
    """
    리스트를 chunk_size 단위로 나눈다.
    예: 56프레임 -> 8개씩 7청크
    """
    return [
        items[i:i + chunk_size]
        for i in range(0, len(items), chunk_size)
    ]

# Cell 6. OpenCV 보조지표 계산 함수

In [9]:
def compute_opencv_stats(image_paths):
    """
    OpenCV 기반 보조지표 계산
    - avg_brightness
    - avg_blue
    - avg_green
    - avg_red
    
    팀원 Gemini 코드 기준과 맞추기 위해 50x50 축소 후 계산
    """
    import cv2

    brightness_values = []
    red_values = []
    green_values = []
    blue_values = []

    for path in image_paths:
        img = cv2.imread(str(path))

        if img is None:
            continue

        img = cv2.resize(img, (50, 50))

        b, g, r = cv2.split(img)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        brightness_values.append(float(np.mean(gray)))
        blue_values.append(float(np.mean(b)))
        green_values.append(float(np.mean(g)))
        red_values.append(float(np.mean(r)))

    if len(brightness_values) == 0:
        return {
            "avg_brightness": None,
            "avg_blue": None,
            "avg_green": None,
            "avg_red": None
        }

    return {
        "avg_brightness": round(float(np.mean(brightness_values)), 3),
        "avg_blue": round(float(np.mean(blue_values)), 3),
        "avg_green": round(float(np.mean(green_values)), 3),
        "avg_red": round(float(np.mean(red_values)), 3)
    }

# Cell 7. JSON 파싱 함수

In [10]:
def extract_json_from_output(output_text):
    """
    모델 출력에서 JSON 부분만 추출
    1. ```json ... ``` 형태
    2. { ... } 형태
    둘 다 대응
    """
    markdown_match = re.search(
        r"```json\s*(\{.*?\})\s*```",
        output_text,
        re.DOTALL
    )

    if markdown_match:
        return markdown_match.group(1)

    json_match = re.search(
        r"\{.*\}",
        output_text,
        re.DOTALL
    )

    if json_match:
        return json_match.group(0)

    return output_text


def parse_json_safely(json_str):
    """
    JSON 파싱 안정화
    """
    try:
        return json.loads(json_str)
    except Exception:
        pass

    try:
        return ast.literal_eval(json_str)
    except Exception:
        pass

    fixed = json_str.replace("'", '"')

    try:
        return json.loads(fixed)
    except Exception as e:
        raise ValueError(f"JSON 파싱 실패: {e}")

# Cell 8. 필수 key 검증 및 색상명 후처리

In [11]:
REQUIRED_KEYS = [
    "production_quality",
    "lighting_style",
    "color_mood",
    "editing_pace",
    "motion_graphic",
    "video_format",
    "first_3sec",
    "background_style",
    "top_colors",
    "person_ratio",
    "face_ratio",
    "text_ratio",
    "reason"
]

DEFAULT_VALUES = {
    "production_quality": "미분류",
    "lighting_style": "미분류",
    "color_mood": "미분류",
    "editing_pace": "미분류",
    "motion_graphic": "미분류",
    "video_format": "기타",
    "first_3sec": "장면",
    "background_style": "혼합",
    "top_colors": [],
    "person_ratio": None,
    "face_ratio": None,
    "text_ratio": None,
    "reason": "판단 근거가 출력되지 않았습니다."
}


def validate_and_fill_result(result):
    """
    필수 key가 누락된 경우 기본값으로 채운다.
    """
    missing_keys = []

    for key in REQUIRED_KEYS:
        if key not in result:
            result[key] = DEFAULT_VALUES[key]
            missing_keys.append(key)

    result["missing_keys"] = missing_keys
    result["is_schema_complete"] = len(missing_keys) == 0

    return result

In [12]:
def normalize_schema_keys(result):
    """
    모델이 key 이름을 약간 틀리게 출력한 경우 보정한다.
    예: color_ mood -> color_mood
    """
    key_aliases = {
        "color_ mood": "color_mood",
        "color mood": "color_mood",
        "colour_mood": "color_mood",

        "production quality": "production_quality",
        "lighting style": "lighting_style",
        "editing pace": "editing_pace",
        "motion graphic": "motion_graphic",
        "video format": "video_format",
        "first 3sec": "first_3sec",
        "background style": "background_style",
        "background_ style": "background_style",
        "top colors": "top_colors",
        "person ratio": "person_ratio",
        "face ratio": "face_ratio",
        "text ratio": "text_ratio",
    }

    normalized = {}

    for key, value in result.items():
        clean_key = key.strip()
        new_key = key_aliases.get(clean_key, clean_key)
        normalized[new_key] = value

    return normalized

In [13]:
COLOR_MAP = {
    # 중국어 → 한국어
    "嫩绿色": "연두색",
    "绿色": "초록색",
    "粉红色": "분홍색",
    "黄色": "노란색",
    "白色": "흰색",
    "黑色": "검은색",
    "红色": "빨간색",
    "蓝色": "파란색",
    "棕色": "갈색",
    "灰色": "회색",
    "橙色": "주황색",
    "紫色": "보라색",

    # 영어 → 한국어
    "green": "초록색",
    "light green": "연두색",
    "pink": "분홍색",
    "yellow": "노란색",
    "white": "흰색",
    "black": "검은색",
    "red": "빨간색",
    "blue": "파란색",
    "brown": "갈색",
    "gray": "회색",
    "grey": "회색",
    "orange": "주황색",
    "purple": "보라색",
    "beige": "베이지색",

    # 한국어 비표준 → 표준
    "화이트": "흰색",
    "검정": "검은색",
    "파랑": "파란색",
    "파랑색": "파란색",
    "빨강": "빨간색",
    "노랑": "노란색",
    "초록": "초록색",
    "핑크": "분홍색",
}

In [14]:
from collections import Counter

def to_float_or_none(value):
    """
    모델이 '0.50'처럼 문자열로 준 비율값을 float로 변환
    """
    try:
        return float(value)
    except Exception:
        return None


def weighted_average(values, weights):
    """
    None을 제외하고 가중 평균 계산
    """
    valid = [
        (v, w)
        for v, w in zip(values, weights)
        if v is not None and w is not None and w > 0
    ]

    if not valid:
        return None

    total_weight = sum(w for _, w in valid)

    if total_weight == 0:
        return None

    return round(sum(v * w for v, w in valid) / total_weight, 2)


def weighted_mode(values, weights):
    """
    청크별 frame 수를 가중치로 둔 최빈값
    모델이 실수로 리스트를 출력한 경우 첫 번째 값만 사용한다.
    """
    counter = Counter()

    for value, weight in zip(values, weights):
        if value is None:
            continue

        if isinstance(value, list):
            value = value[0] if value else None

        if value is not None:
            counter[value] += weight

    if not counter:
        return "미분류"

    return counter.most_common(1)[0][0]


def aggregate_chunk_results(chunk_results, original_frame_count):
    """
    여러 청크의 VLM 결과를 영상 1개 결과로 통합
    """
    if len(chunk_results) == 0:
        raise ValueError("통합할 chunk_results가 없습니다.")

    weights = [
        result.get("used_frame_count", 1)
        for result in chunk_results
    ]

    # 범주형 컬럼: 프레임 수 가중 최빈값
    categorical_keys = [
        "production_quality",
        "lighting_style",
        "color_mood",
        "editing_pace",
        "motion_graphic",
        "video_format",
        "background_style",
    ]

    final = {}

    for key in categorical_keys:
        final[key] = weighted_mode(
            [result.get(key) for result in chunk_results],
            weights
        )

    # first_3sec는 전체 영상의 첫 3초 기준이므로 첫 번째 청크 결과를 우선 사용
    final["first_3sec"] = chunk_results[0].get("first_3sec", "장면")

    # 비율 컬럼: 청크 프레임 수 가중 평균
    for key in ["person_ratio", "face_ratio", "text_ratio"]:
        values = [
            to_float_or_none(result.get(key))
            for result in chunk_results
        ]

        final[key] = weighted_average(values, weights)

    # top_colors: 모든 청크 색상 중 가장 많이 나온 3개
    color_counter = Counter()

    for result in chunk_results:
        colors = result.get("top_colors", [])

        if isinstance(colors, str):
            colors = [colors]

        for color in colors:
            if color and color != "기타":
                color_counter[color] += 1

    top_colors = [
        color
        for color, _ in color_counter.most_common(3)
    ]

    while len(top_colors) < 3:
        top_colors.append("기타")

    final["top_colors"] = top_colors[:3]

    # reason: 너무 길게 합치지 않고 요약형으로 생성
    final["reason"] = (
        f"{len(chunk_results)}개 구간의 분석 결과를 종합했으며, "
        f"{final['video_format']} 형식과 {final['background_style']} 배경 특징이 두드러진다."
    )

    # 메타 정보
    final["model_name"] = "HyperCLOVAX-SEED-Vision-Instruct-3B"
    final["frame_sampling_method"] = "1fps_chunked_all_frames"
    final["used_frame_count"] = original_frame_count
    final["original_frame_count"] = original_frame_count
    final["chunk_count"] = len(chunk_results)

    return final

In [15]:
def normalize_top_colors(result):
    """
    top_colors에 중국어/영어/비표준 색상명이 섞인 경우 한국어 표준명으로 변환한다.
    3개보다 적으면 '기타'로 채운다.
    """
    colors = result.get("top_colors", [])

    if isinstance(colors, str):
        colors = [colors]

    normalized = []

    for color in colors:
        color_str = str(color).strip()
        color_lower = color_str.lower()

        if color_str in COLOR_MAP:
            normalized.append(COLOR_MAP[color_str])
        elif color_lower in COLOR_MAP:
            normalized.append(COLOR_MAP[color_lower])
        else:
            normalized.append(color_str)

    # 중복 제거
    normalized = list(dict.fromkeys(normalized))

    # 3개 미만이면 보정
    while len(normalized) < 3:
        normalized.append("기타")

    result["top_colors"] = normalized[:3]

    return result

# HyperCLOVAX 입력 텐서를 GPU로 옮기고, 이미지/실수 텐서는 float16으로 변환

In [16]:
from collections.abc import Mapping

def move_to_device(obj, device="cuda", dtype=torch.float16):
    """
    dict, BatchEncoding, BatchFeature, list, tuple 내부 tensor까지 모두 device로 이동.
    정수 tensor(input_ids, attention_mask 등)는 dtype 유지.
    실수/이미지/uint8 tensor는 float16으로 변환.
    """
    if torch.is_tensor(obj):
        if torch.is_floating_point(obj) or obj.dtype == torch.uint8:
            return obj.to(device=device, dtype=dtype)
        else:
            return obj.to(device=device)

    # BatchEncoding / BatchFeature / dict 모두 처리
    if isinstance(obj, Mapping) or hasattr(obj, "items"):
        return {
            k: move_to_device(v, device=device, dtype=dtype)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [
            move_to_device(v, device=device, dtype=dtype)
            for v in obj
        ]

    if isinstance(obj, tuple):
        return tuple(
            move_to_device(v, device=device, dtype=dtype)
            for v in obj
        )

    return obj

# Cell 9. 프롬프트 정의

In [ ]:
PROMPT_TEXT = """
너는 기업 유튜브 쇼츠 영상을 분석하는 비주얼 마케팅 전문가다.
주어진 이미지들은 하나의 쇼츠 영상에서 1초 단위로 추출된 일부 프레임이다.

반드시 순수 JSON 객체 1개만 출력한다.
마크다운 코드블록, 설명문, 주석은 출력하지 않는다.
모든 값은 한국어로 작성한다.
아래 key 이름을 절대 바꾸지 않는다.

중요 규칙:
- 선택지 목록 전체를 출력하지 않는다.
- 각 항목은 반드시 선택지 중 하나의 문자열 값만 출력한다.
- 배열로 출력하는 항목은 top_colors만 허용한다.
- top_colors는 정확히 색상명 3개 리스트로 출력한다.
- person_ratio, face_ratio, text_ratio는 0.00~1.00 사이 숫자로 출력한다.
- 반드시 { 로 시작하고 } 로 끝나는 JSON만 출력한다.

선택지:
production_quality = 저예산, 일반, 고퀄리티, 프로페셔널
lighting_style = 자연광, 인공조명, 역광, 저조도, 혼합
color_mood = 따뜻함, 차가움, 중립, 비비드, 무채색
editing_pace = 매우 느림, 느림, 보통, 빠름, 매우 빠름
motion_graphic = 없음, 보조적, 핵심요소
video_format = 웹드라마, 브이로그, 시설소개, 에피소드소개, 제품리뷰, 튜토리얼, 광고/CF, 다큐멘터리, 웹예능, 이벤트/행사, 인터뷰, 애니메이션, 기술설명, 요리/레시피, 영양정보, 고객후기, 기타
first_3sec = 텍스트, 인물, 제품, 장면, 기업 로고, 음식
background_style = 실내, 실외, 스튜디오, 혼합

판단 기준:
- video_format은 등장 소재보다 영상 목적을 우선한다.
- 음식이 등장해도 조리 과정 설명이 아니라 제품/브랜드 홍보 목적이면 광고/CF로 분류한다.
- 요리/레시피는 실제 조리 과정이나 레시피 안내가 중심일 때만 선택한다.
- person_ratio는 입력된 프레임 중 인물이 등장하는 프레임 비율이다.
- face_ratio는 입력된 프레임 중 얼굴이 명확히 보이는 프레임 비율이다.
- text_ratio는 입력된 프레임 중 텍스트/자막이 보이는 프레임 비율이다.
- reason은 주요 분류 결과를 뒷받침하는 시각적 근거를 1문장으로 작성한다.
"""

# Cell 10. HyperCLOVAX 영상 1개 분석 함수

In [18]:
from collections.abc import Mapping

def print_tensor_devices(obj, prefix="model_inputs"):
    """
    nested structure 안의 tensor dtype/device 확인
    """
    if torch.is_tensor(obj):
        print(prefix, obj.dtype, tuple(obj.shape), obj.device)
        return

    if isinstance(obj, Mapping) or hasattr(obj, "items"):
        for k, v in obj.items():
            print_tensor_devices(v, f"{prefix}.{k}")
        return

    if isinstance(obj, list):
        for i, v in enumerate(obj):
            print_tensor_devices(v, f"{prefix}[{i}]")
        return

    if isinstance(obj, tuple):
        for i, v in enumerate(obj):
            print_tensor_devices(v, f"{prefix}({i})")
        return

In [19]:
def infer_one_hyperclovax(
    model,
    processor,
    video_id,
    image_paths,
    max_new_tokens=384,
    max_retries=1,
    debug=False
):
    if len(image_paths) == 0:
        raise ValueError(f"{video_id}: 입력 이미지가 없습니다.")

    content = []

    for path in image_paths:
        content.append({
            "type": "image",
            "filename": path.name,
            "image": str(path)
        })

    content.append({
        "type": "text",
        "text": PROMPT_TEXT
    })

    vlm_chat = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "너는 기업 유튜브 쇼츠 영상을 분석하는 비주얼 마케팅 전문가다."
                }
            ]
        },
        {
            "role": "user",
            "content": content
        }
    ]

    last_output = ""

    for attempt in range(max_retries):
        torch.cuda.empty_cache()
        gc.collect()

        try:
            model_inputs = processor.apply_chat_template(
                vlm_chat,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True
            )

            if debug:
                print("===== processor 출력 확인 =====")
                print_tensor_devices(model_inputs)
                print("==============================")

            model_inputs = move_to_device(
                model_inputs,
                device="cuda",
                dtype=torch.float16
            )

            if debug:
                print("===== move_to_device 이후 확인 =====")
                print_tensor_devices(model_inputs)
                print("===============================")

            start_time = time.time()

            with torch.no_grad():
                output_ids = model.generate(
                    **model_inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.1
                )

            elapsed = time.time() - start_time

            full_output_text = processor.batch_decode(
                output_ids,
                skip_special_tokens=True
            )[0]

            input_len = model_inputs["input_ids"].shape[1]
            generated_only = output_ids[:, input_len:]

            output_text = processor.batch_decode(
                generated_only,
                skip_special_tokens=True
            )[0]

            if not output_text.strip():
                output_text = full_output_text

            last_output = output_text

            if debug:
                print("===== output_text =====")
                print(output_text[:1000])
                print("=======================")

            json_str = extract_json_from_output(output_text)
            result = parse_json_safely(json_str)

            result = normalize_schema_keys(result)
            result = validate_and_fill_result(result)
            result = normalize_top_colors(result)

            result["video_id"] = video_id
            result["model_name"] = "HyperCLOVAX-SEED-Vision-Instruct-3B"
            result["frame_sampling_method"] = "1fps_chunk"
            result["used_frame_count"] = len(image_paths)
            result["inference_seconds"] = round(elapsed, 2)

            del model_inputs
            del output_ids
            torch.cuda.empty_cache()
            gc.collect()

            return result

        except Exception as e:
            torch.cuda.empty_cache()
            gc.collect()

            print(f"[{video_id}] HyperCLOVAX 분석 실패 {attempt + 1}/{max_retries}")
            print("에러:", str(e)[:500])
            print("출력 일부:", last_output[:500])

            if attempt == max_retries - 1:
                raise

In [20]:
def infer_video_hyperclovax_chunked(
    model,
    processor,
    video_id,
    image_paths,
    chunk_size=8,
    max_new_tokens=256,
    debug=False
):
    """
    하나의 영상 전체 프레임을 chunk 단위로 나눠 분석하고,
    chunk 결과를 하나의 영상 결과로 통합한다.
    """
    if len(image_paths) == 0:
        raise ValueError(f"{video_id}: 입력 이미지가 없습니다.")

    chunks = split_into_chunks(image_paths, chunk_size)

    print(f"{video_id} | 전체 프레임 수: {len(image_paths)} | 청크 수: {len(chunks)}")

    chunk_results = []

    for chunk_idx, chunk_paths in enumerate(chunks):
        print(f"  - 청크 {chunk_idx + 1}/{len(chunks)} 분석 중 | 프레임 수: {len(chunk_paths)}")

        try:
            chunk_result = infer_one_hyperclovax(
                model=model,
                processor=processor,
                video_id=f"{video_id}_chunk_{chunk_idx + 1}",
                image_paths=chunk_paths,
                max_new_tokens=max_new_tokens,
                max_retries=1,
                debug=debug
            )

            chunk_result["chunk_index"] = chunk_idx + 1
            chunk_result["chunk_frame_count"] = len(chunk_paths)

            chunk_results.append(chunk_result)

        except RuntimeError as e:
            # 청크도 OOM이면 chunk_size를 더 줄여야 함
            torch.cuda.empty_cache()
            gc.collect()

            print(f"  ❌ 청크 {chunk_idx + 1} 실패: {str(e)[:300]}")
            raise

        finally:
            torch.cuda.empty_cache()
            gc.collect()

    final_result = aggregate_chunk_results(
        chunk_results=chunk_results,
        original_frame_count=len(image_paths)
    )

    final_result["video_id"] = video_id

    return final_result

# Cell 11. 영상 1개 Dry Run 준비

In [21]:
# video_id = video_dirs[0].name
# image_paths = gather_images(FRAMES_ROOT / video_id)

# # 1fps 전체 프레임 사용
# used_paths = image_paths

# print("video_id:", video_id)
# print("사용 프레임 수:", len(used_paths))

# for p in used_paths[:5]:
#     print(p)

In [22]:
video_id = video_dirs[0].name
image_paths = gather_images(FRAMES_ROOT / video_id)

# 디버그용: 먼저 이미지 1장만 테스트
used_paths = image_paths[:1]

print("video_id:", video_id)
print("전체 프레임 수:", len(image_paths))
print("사용 프레임 수:", len(used_paths))
print(used_paths[0])

video_id: knCQBlBKSRM
전체 프레임 수: 21
사용 프레임 수: 1
data\frames_for_vlm\knCQBlBKSRM\frame_000.jpg


# Cell 12. 영상 1개 Dry Run 실행

In [23]:
hyper_result = infer_one_hyperclovax(
    model=model,
    processor=processor,
    video_id=video_id,
    image_paths=used_paths,
    max_new_tokens=MAX_NEW_TOKENS,
    max_retries=3
)

opencv_stats = compute_opencv_stats(used_paths)

final_result = {
    **hyper_result,
    **opencv_stats,
    "original_frame_count": len(image_paths),
    "used_frame_count": len(used_paths),
    "frame_sampling_method": "1fps_all_frames"
}

# 최종 제출/비교용에서는 디버깅 컬럼 제거
final_result.pop("missing_keys", None)
final_result.pop("is_schema_complete", None)

print(json.dumps(final_result, ensure_ascii=False, indent=2))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "production_quality": "일반",
  "lighting_style": "자연광",
  "color_mood": "중립",
  "editing_pace": "보통",
  "motion_graphic": "없음",
  "video_format": "광고/CF",
  "first_3sec": "텍스트",
  "background_style": "실외",
  "top_colors": [
    "흰색",
    "검은색",
    "기타"
  ],
  "person_ratio": "0.50",
  "face_ratio": "0.30",
  "text_ratio": "0.70",
  "reason": "텍스트와 제품 노출이 중심이라 광고성 영상으로 판단했다.",
  "video_id": "knCQBlBKSRM",
  "model_name": "HyperCLOVAX-SEED-Vision-Instruct-3B",
  "frame_sampling_method": "1fps_all_frames",
  "used_frame_count": 1,
  "inference_seconds": 53.51,
  "avg_brightness": 49.536,
  "avg_blue": 44.339,
  "avg_green": 47.93,
  "avg_red": 54.656,
  "original_frame_count": 21
}


# Cell 13. 전체 영상 자동 분석

In [24]:
all_results = []
failed_results = []

# 기존 결과 파일 삭제 후 새로 시작
if CSV_PATH.exists():
    CSV_PATH.unlink()

if JSON_PATH.exists():
    JSON_PATH.unlink()

if FAILED_PATH.exists():
    FAILED_PATH.unlink()

print("HyperCLOVAX 청크 기반 전체 영상 분석 시작")
print("분석 대상 개수:", len(video_dirs))
print([p.name for p in video_dirs])

for idx, video_dir in enumerate(video_dirs):
    video_id = video_dir.name

    print(f"\n[{idx + 1}/{len(video_dirs)}] 분석 중: {video_id}")

    try:
        image_paths = gather_images(video_dir)

        print(f"전체 프레임 수: {len(image_paths)}")
        print(f"청크 크기: {CHUNK_SIZE}")

        vlm_result = infer_video_hyperclovax_chunked(
            model=model,
            processor=processor,
            video_id=video_id,
            image_paths=image_paths,
            chunk_size=CHUNK_SIZE,
            max_new_tokens=MAX_NEW_TOKENS,
            debug=False
        )

        # OpenCV는 전체 프레임 기준으로 한 번만 계산
        opencv_stats = compute_opencv_stats(image_paths)

        final_result = {
            **vlm_result,
            **opencv_stats,
            "original_frame_count": len(image_paths),
            "used_frame_count": len(image_paths),
            "frame_sampling_method": "1fps_chunked_all_frames"
        }

        # 최종 비교용에서는 디버깅 컬럼 제거
        final_result.pop("missing_keys", None)
        final_result.pop("is_schema_complete", None)

        all_results.append(final_result)

        df_result = pd.DataFrame([final_result])

        if not CSV_PATH.exists():
            df_result.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
        else:
            df_result.to_csv(
                CSV_PATH,
                mode="a",
                header=False,
                index=False,
                encoding="utf-8-sig"
            )

        print(f"✅ 분석 완료 | chunk_count: {final_result.get('chunk_count')}")

    except Exception as e:
        print(f"❌ 분석 실패: {video_id}")
        print(e)

        failed_results.append({
            "video_id": video_id,
            "error": str(e)
        })

        torch.cuda.empty_cache()
        gc.collect()

        continue

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

with open(FAILED_PATH, "w", encoding="utf-8") as f:
    json.dump(failed_results, f, ensure_ascii=False, indent=2)

print("\n전체 작업 완료")
print("성공 개수:", len(all_results))
print("실패 개수:", len(failed_results))
print("CSV 저장 위치:", CSV_PATH)
print("JSON 저장 위치:", JSON_PATH)
print("실패 로그 저장 위치:", FAILED_PATH)

if failed_results:
    print("\n실패 목록:")
    for item in failed_results:
        print(item["video_id"], "->", item["error"][:300])

HyperCLOVAX 청크 기반 전체 영상 분석 시작
분석 대상 개수: 8
['knCQBlBKSRM', 'LRmLOsmMHts', 'mfi0n3oNABM', 'MIJR3Dm0YOk', 'NCVxpXxTMSU', 'twi9zUxsXu0', 'Wkiz8JVPvXA', 'Xfu1kCID0Ls']

[1/8] 분석 중: knCQBlBKSRM
전체 프레임 수: 21
청크 크기: 8
knCQBlBKSRM | 전체 프레임 수: 21 | 청크 수: 3
  - 청크 1/3 분석 중 | 프레임 수: 8


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  - 청크 2/3 분석 중 | 프레임 수: 8


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  - 청크 3/3 분석 중 | 프레임 수: 5


...

  - 청크 2/3 분석 중 | 프레임 수: 8


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  - 청크 3/3 분석 중 | 프레임 수: 7


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ 분석 완료 | chunk_count: 3

전체 작업 완료
성공 개수: 8
실패 개수: 0
CSV 저장 위치: results\hyperclovax_vlm_1fps_chunked_results.csv
JSON 저장 위치: results\hyperclovax_vlm_1fps_chunked_results.json
실패 로그 저장 위치: results\hyperclovax_vlm_1fps_chunked_failed.json


# Cell 14. 결과 확인

In [27]:
if CSV_PATH.exists():
    df_hyper = pd.read_csv(CSV_PATH)

    print("결과 shape:", df_hyper.shape)
    display(df_hyper.head())
else:
    print("CSV 파일이 아직 없습니다:", CSV_PATH)

    if FAILED_PATH.exists():
        print("실패 로그 파일은 존재합니다:", FAILED_PATH)

결과 shape: (8, 23)


,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,video_format,background_style,first_3sec,person_ratio,face_ratio,...,model_name,frame_sampling_method,used_frame_count,original_frame_count,chunk_count,video_id,avg_brightness,avg_blue,avg_green,avg_red
0,일반,자연광,중립,보통,없음,광고/CF,실내,텍스트,0.5,0.3,...,HyperCLOVAX-SEED-Vision-Instruct-3B,1fps_chunked_all_frames,21,21,3,knCQBlBKSRM,34.737,28.688,34.006,38.514
1,일반,자연광,중립,보통,없음,광고/CF,실내,텍스트,0.5,0.3,...,HyperCLOVAX-SEED-Vision-Instruct-3B,1fps_chunked_all_frames,30,30,4,LRmLOsmMHts,195.093,196.727,193.490,197.750
2,프로페셔널,자연광,비비드,보통,없음,광고/CF,실외,인물,0.5,0.3,...,HyperCLOVAX-SEED-Vision-Instruct-3B,1fps_chunked_all_frames,56,56,7,mfi0n3oNABM,115.403,107.136,120.163,109.228
3,일반,자연광,중립,보통,없음,광고/CF,실내,인물,0.5,0.3,...,HyperCLOVAX-SEED-Vision-Instruct-3B,1fps_chunked_all_frames,50,50,7,MIJR3Dm0YOk,26.621,25.146,25.716,28.958
4,일반,자연광,중립,보통,없음,광고/CF,실내,텍스트,0.5,0.3,...,HyperCLOVAX-SEED-Vision-Instruct-3B,1fps_chunked_all_frames,12,12,2,NCVxpXxTMSU,59.565,46.589,58.174,67.237


In [29]:
check_cols = [
    "video_id",
    "model_name",
    "video_format",
    "production_quality",
    "lighting_style",
    "color_mood",
    "editing_pace",
    "motion_graphic",
    "first_3sec",
    "background_style",
    "top_colors",
    "person_ratio",
    "face_ratio",
    "text_ratio",
    "reason",    
    "used_frame_count"
]

df_hyper[check_cols]

,video_id,model_name,video_format,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,first_3sec,background_style,top_colors,person_ratio,face_ratio,text_ratio,reason,used_frame_count
0,knCQBlBKSRM,HyperCLOVAX-SEED-Vision-Instruct-3B,광고/CF,일반,자연광,중립,보통,없음,텍스트,실내,"['흰색', '검은색', '파란색']",0.5,0.3,0.7,"3개 구간의 분석 결과를 종합했으며, 광고/CF 형식과 실내 배경 특징이 두드러진다.",21
1,LRmLOsmMHts,HyperCLOVAX-SEED-Vision-Instruct-3B,광고/CF,일반,자연광,중립,보통,없음,텍스트,실내,"['흰색', '검은색', '파란색']",0.5,0.3,0.7,"4개 구간의 분석 결과를 종합했으며, 광고/CF 형식과 실내 배경 특징이 두드러진다.",30
2,mfi0n3oNABM,HyperCLOVAX-SEED-Vision-Instruct-3B,광고/CF,프로페셔널,자연광,비비드,보통,없음,인물,실외,"['흰색', '검은색', '파란색']",0.5,0.3,0.7,"7개 구간의 분석 결과를 종합했으며, 광고/CF 형식과 실외 배경 특징이 두드러진다.",56
3,MIJR3Dm0YOk,HyperCLOVAX-SEED-Vision-Instruct-3B,광고/CF,일반,자연광,중립,보통,없음,인물,실내,"['검은색', '흰색', '파란색']",0.5,0.3,0.7,"7개 구간의 분석 결과를 종합했으며, 광고/CF 형식과 실내 배경 특징이 두드러진다.",50
4,NCVxpXxTMSU,HyperCLOVAX-SEED-Vision-Instruct-3B,광고/CF,일반,자연광,중립,보통,없음,텍스트,실내,"['흰색', '검은색', '파란색']",0.5,0.3,0.7,"2개 구간의 분석 결과를 종합했으며, 광고/CF 형식과 실내 배경 특징이 두드러진다.",12
5,twi9zUxsXu0,HyperCLOVAX-SEED-Vision-Instruct-3B,광고/CF,프로페셔널,자연광,중립,보통,없음,인물,실내,"['흰색', '검은색', '파란색']",0.5,0.3,0.7,"7개 구간의 분석 결과를 종합했으며, 광고/CF 형식과 실내 배경 특징이 두드러진다.",55
6,Wkiz8JVPvXA,HyperCLOVAX-SEED-Vision-Instruct-3B,광고/CF,프로페셔널,자연광,중립,보통,없음,음식,실내,"['빨간색', '노란색', '흰색']",0.5,0.3,0.7,"4개 구간의 분석 결과를 종합했으며, 광고/CF 형식과 실내 배경 특징이 두드러진다.",30
7,Xfu1kCID0Ls,HyperCLOVAX-SEED-Vision-Instruct-3B,광고/CF,고퀄리티,자연광,중립,보통,없음,인물,실내,"['흰색', '검은색', '파란색']",0.5,0.3,0.7,"3개 구간의 분석 결과를 종합했으며, 광고/CF 형식과 실내 배경 특징이 두드러진다.",23


# Cell 15. GPU 메모리 정리

In [30]:
del model
del processor
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print("HyperCLOVAX 모델 삭제 및 CUDA cache 정리 완료")

HyperCLOVAX 모델 삭제 및 CUDA cache 정리 완료
